# Notebook 02: BEVFormer — 3D Detection with Cameras Only

**Goal**: Understand how BEVFormer converts multi-camera images into 3D detections using Bird's Eye View representation.

## Architecture Overview

```
6 cameras (front, front-left, front-right, back, back-left, back-right)
      ↓ ResNet-101 + FPN
Multi-scale image features [B, N_cam, C, H, W]
      ↓ Spatial Cross-Attention
BEV Features [B, bev_h×bev_w, C]  (200×200 grid, 0.512m per cell)
      ↓ Temporal Self-Attention
Temporally-fused BEV
      ↓ DETR Detection Head
3D Boxes [cx, cy, cz, w, l, h, yaw, vx, vy] × num_classes
```

## Key Innovations
1. **BEV Queries**: Learned BEV grid → no depth prediction needed
2. **Spatial Cross-Attention**: 3D reference points project to camera planes
3. **Temporal Attention**: Previous frame BEV aligned via ego-motion

In [ ]:
import sys
sys.path.insert(0, '..')

# Setup: clone BEVFormer repo first
# !git clone https://github.com/fundamentalvision/BEVFormer.git ../external/BEVFormer
# sys.path.insert(0, '../external/BEVFormer')

print('BEVFormer notebook ready')
print('Requires: cloned BEVFormer repo + nuScenes dataset')

## Understanding Spatial Cross-Attention

The most important component of BEVFormer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_bev_to_camera_projection():
    """
    Show how BEV reference points (3D grid) project to camera images.
    This is the core of Spatial Cross-Attention.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # ── Left: BEV grid ──────────────────────────────────────────────────
    ax_bev = axes[0]
    bev_size = 200
    bev_range = 51.2  # meters

    ax_bev.set_xlim(-bev_range, bev_range)
    ax_bev.set_ylim(-bev_range, bev_range)
    ax_bev.set_facecolor('#1a1a2e')
    ax_bev.set_title('BEV Grid (200×200 queries)', fontsize=12)
    ax_bev.set_xlabel('X (meters)')
    ax_bev.set_ylabel('Y (meters)')

    # Draw grid cells
    cell_size = 2 * bev_range / bev_size
    for i in range(-5, 6):
        for j in range(-5, 6):
            x = i * cell_size * 5
            y = j * cell_size * 5
            ax_bev.add_patch(patches.Rectangle(
                (x - cell_size/2, y - cell_size/2), cell_size, cell_size,
                fill=True, facecolor='#16213e', edgecolor='#0f3460', linewidth=0.5,
            ))

    # Ego vehicle
    ax_bev.add_patch(patches.Rectangle((-2, -3), 4, 6, fill=True, color='#00ff87'))
    ax_bev.text(0, 0, 'EGO', ha='center', va='center', color='black', fontweight='bold')

    # Sample 3D reference points (one per BEV query pillar)
    ref_x = np.random.uniform(-30, 30, 20)
    ref_y = np.random.uniform(-30, 30, 20)
    ref_z = np.array([-1.0, -0.5, 0.0, 1.0])  # 4 heights per pillar

    for x, y in zip(ref_x, ref_y):
        ax_bev.plot(x, y, 'o', color='#ff6b6b', markersize=4)

    # Camera FOV (front camera)
    fov_angle = np.radians(70)  # ~70° FOV
    for angle in [fov_angle/2, -fov_angle/2]:
        ax_bev.plot([0, 50*np.sin(angle)], [0, 50*np.cos(angle)],
                    '--', color='#ffa07a', alpha=0.6, linewidth=1.5)
    ax_bev.text(5, 35, 'CAM_FRONT\nFOV', color='#ffa07a', fontsize=8)

    # Highlight selected query
    qx, qy = ref_x[0], ref_y[0]
    ax_bev.plot(qx, qy, '*', color='yellow', markersize=15, zorder=5)
    ax_bev.annotate('Query → projects to\ncamera image', (qx, qy),
                    xytext=(qx+8, qy+5), color='yellow', fontsize=8,
                    arrowprops=dict(arrowstyle='->', color='yellow'))

    # ── Right: Camera image projection ──────────────────────────────────
    ax_cam = axes[1]
    ax_cam.set_xlim(0, 1600)
    ax_cam.set_ylim(900, 0)  # image coords (y down)
    ax_cam.set_facecolor('#1a1a2e')
    ax_cam.set_title('Camera Image (Spatial Cross-Attention)', fontsize=12)
    ax_cam.set_xlabel('Image X (pixels)')
    ax_cam.set_ylabel('Image Y (pixels)')

    # Simulated image with cars
    ax_cam.add_patch(patches.Rectangle((0, 0), 1600, 900, fill=True, facecolor='#16213e'))
    ax_cam.add_patch(patches.Rectangle((0, 400), 1600, 500, fill=True, facecolor='#2d2d2d', label='road'))
    ax_cam.add_patch(patches.Rectangle((300, 350), 200, 150, fill=True, facecolor='#cc3333', label='car1'))
    ax_cam.add_patch(patches.Rectangle((700, 320), 300, 180, fill=True, facecolor='#3333cc', label='truck'))
    ax_cam.add_patch(patches.Rectangle((1100, 380), 180, 120, fill=True, facecolor='#33cc33', label='car2'))

    # Projected reference points (deformable attention sampling points)
    proj_x = np.array([380, 400, 420, 760, 780, 800, 1160, 1180])
    proj_y = np.array([400, 420, 450, 380, 400, 430, 410, 440])
    ax_cam.scatter(proj_x, proj_y, c='yellow', s=80, zorder=5, label='sampling points')
    ax_cam.text(400, 300, 'Deformable attention\nsampling points', color='yellow', fontsize=9)

    # Highlighted projected query
    ax_cam.plot(400, 420, '*', color='yellow', markersize=20, zorder=6)
    ax_cam.annotate('Same query as BEV★', (400, 420),
                    xytext=(200, 250), color='yellow', fontsize=9,
                    arrowprops=dict(arrowstyle='->', color='yellow'))

    plt.tight_layout()
    plt.savefig('../runs/bevformer_attention_viz.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: runs/bevformer_attention_viz.png')

import os
os.makedirs('../runs', exist_ok=True)
visualize_bev_to_camera_projection()

## Training BEVFormer on HPC

Use the SLURM script directly — BEVFormer training is compute-heavy.

In [ ]:
from models.detector_3d import BEVFormerDetector

# Show the training command for HPC
cmd = BEVFormerDetector.get_train_command(
    config='configs/bevformer/bevformer_base.py',
    gpus=8,
    work_dir='work_dirs/bevformer_base',
)
print('Training command:')
print(cmd)
print()
print('SLURM submission:')
print('  sbatch training/slurm/bevformer_train.sh')

## Understanding nuScenes Metrics

nuScenes uses custom metrics different from COCO mAP:

In [ ]:
# nuScenes Detection Score (NDS) breakdown
nds_components = {
    'mAP':   (0.3, 'Mean Average Precision (matching threshold = 2m center distance)'),
    'mATE':  (0.25, 'Mean Avg Translation Error (meters) — lower is better'),
    'mASE':  (0.25, 'Mean Avg Scale Error (1 - IoU) — lower is better'),
    'mAOE':  (0.25, 'Mean Avg Orientation Error (radians) — lower is better'),
    'mAVE':  (0.0, 'Mean Avg Velocity Error (m/s) — lower is better'),
    'mAAE':  (0.0, 'Mean Avg Attribute Error — lower is better'),
}

print('NDS = 0.5 * mAP + 0.5 * mean(1-TP_errors)')
print('where TP errors are: ATE, ASE, AOE, AVE, AAE')
print()
print('Component      Weight   Description')
print('-' * 70)
for name, (weight, desc) in nds_components.items():
    print(f'{name:<14} {weight:.2f}     {desc}')

print()
print('SOTA scores on nuScenes val (2024):')
print('  BEVFormer-Base:  NDS=51.7, mAP=41.6 (camera-only)')
print('  BEVFusion:       NDS=72.9, mAP=70.2 (LiDAR+Camera)')
print('  Sparse4D v3:     NDS=67.7, mAP=63.0 (camera-only, temporal)')